GET请求接口
1. 在urls.py 配置请求路径
2. 在view.py 配置请求逻辑

配置返回数据
1. 一个接口可以不用传参数，但是必须要有返回数据
2. return HttpResponse 对象，或 JsonResponse对象

参数配置注意事项：
1. request必须要
2. 一定要指定请求方法
3. 一定要有返回数据
4. 配置url时，记得导入views模块


如何在返回的jsonresponse中设置http的响应状态码，记住非我们定义的code

In [ ]:
from django.views.decorators.http import require_http_methods
from django.http import JsonResponse, HttpResponse

@require_http_methods(['GET'])
def get_api(requeest):
    
    i = 1
    if i == 1:
        return JsonResponse(
            {'msg':'i == 1'},
            status = 200
        )
    else:
        return JsonResponse(
            {'msg':'i != 1'},
            status = 404
        )

1. return JsonResponse(data, status=200) # Django 接口开发的最常见的返回格式 data是返回给前端/request的Json数据，status是HTTP响应码

2. 为什么JsonResponse在IDE自动编译的时候没有status参数：
因为：JsonResponse 本身没有显式定义 status 参数，是继承的父类 HttpResponse.__init__()的构造参数 (super.__init__(...))
-> JsonResponse.__init__() 通过 **kwargs 接收额外参数，并传递给父类 HttpResponse, status 参数实际定义在 HttpResponse 中。

3. JsonResponse.status_code 不能通过类直接访问，不是设置响应码的正确方式 (status_code 是JsonResponse的实例属性，如果要赋值，需要先实例化)
   但：
   response = JsonResponse({"msg": "ok"})
   response.status_code = 404
   return response
   -> 这种写法并不推荐，status_code 是 HttpResponse 内部状态，应该在创建的时候就设置，不要事后改。

   当请求接口 -> response = requests.get('url...')
               respnse.status_code # 200 / 404 / 301 .. 才用到 -> 这是常规用法，多用于接口测试

HttpResponse类

1. HttpResponse是Django最基础的响应类
   作用：构造HTTP响应报文（状态吗、响应头、响应体）
   向前端返回文本、HTML、二进制文件等内容

2. HttpResponse的核心构造参数
   
   HttpResponse(content = b'',status=200,content_type = 'text/html')

   - content 响应体，字符串/字节流
   - status http 状态码，200 成功、3xx 重定向、404 找不到服务，403 参数错误、500服务器错误等
   - content_type 响应头 Content-Type, text/plain 纯文本、application/json Json格式，application/octet-stream 文件下载

3. HttpResponse 设置/获取响应头
    resp = HttpResponse("ok")
    # 设置自定义响应头
    resp["Token"] = "abc123xyz"
    resp["X-User-ID"] = "10086"
    # 删除响应头
    del resp["Token"]
    # 获取响应头
    resp["X-User-ID"]

4. HttpResponse常用子类
   - JsonResponse 返回Json
   def api(request):
    return JsonResponse("msg":"ok"},status = 200) # status 继承自HttpResponse

   - HttpResponseRedirect 重定向 
   from django.shortcuts import redirect
    def to_login(request):
        return redirect("/login/")
   
   - HttpResponseNotFound 404 页面
    return HttpResponseNotFound("页面不存在")

5. HttpResponse 调用
   
   resp = HttpResponse("hello")
   resp.content # b"hello"
   resp.status_code # 状态码
   resp.hearders # 所有响应头字典
   

In [ ]:
# 实现代码
# views.py
from django.http import JsonResponse, HttpResponse
import json

@require_http_methods(["POST"])
def demo_view(request):
    # 1. 构造响应
    data = {"msg": "Demo_View", "user": "Ming"}
    resp = HttpResponse(
        json.dumps(data),
        status=200,
        content_type="application/json; charset=utf-8"
    )
    # 2. 自定义响应头
    resp["X-User-ID"] = "1011"
    # 3. 设置cookie
    resp.set_cookie("session_id", "sdf123456", max_age=86400)
    return resp

In [ ]:
# urls.py
from django.contrib import admin
from django.urls import path
from learndjango import views

urlpatterns = [
    path('admin/', admin.site.urls),
    path('allStuInfo/',views.all_stu_info), # 后续如果调用到allStuInfo 这个接口，就会自动执行 views.py 的all_stu_info方法
    path('createStuInfo/',views.create_stu_info),
    path('demo_view/',views.demo_view)
]

postman response ：
{
    "msg": "Demo_View",
    "user": "Ming"
}

配置headers（读取前端请求头）
request.META